# BÀI THỰC HÀNH HADOOP ONLINE: PHÂN TÍCH ĐƠN HÀNG THƯƠNG MẠI ĐIỆN TỬ VỚI HDFS VÀ MAPREDUCE STREAMING


### 1. Kiểm tra môi trường Java và cài đặt Apache Hadoop 3.5.0


In [ ]:
!java -version
!apt-get update -qq
!apt-get install -y openjdk-17-jdk-headless
!wget -q --show-progress https://archive.apache.org/dist/hadoop/common/hadoop-3.5.0/hadoop-3.5.0.tar.gz
!tar -xzf hadoop-3.5.0.tar.gz


### 2. Thiết lập biến môi trường cho tiến trình Notebook


In [ ]:
import os
import subprocess

os.environ["HADOOP_HOME"] = "/content/hadoop-3.5.0"
java_bin = subprocess.check_output(["bash", "-lc", "dirname $(dirname $(readlink -f $(which java)))"], text=True).strip()
os.environ["JAVA_HOME"] = os.path.realpath(java_bin)
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin:" + os.environ["HADOOP_HOME"] + "/sbin:" + os.environ["PATH"]

with open(os.environ["HADOOP_HOME"] + "/etc/hadoop/hadoop-env.sh", "a") as f:
    f.write(f"\nexport JAVA_HOME={os.environ['JAVA_HOME']}\n")

!hadoop version


### 3. Cấu hình Hadoop Pseudo-Distributed (core-site.xml, hdfs-site.xml, mapred-site.xml)


In [ ]:
!mkdir -p /content/hadoop_data/namenode
!mkdir -p /content/hadoop_data/datanode

core_xml = """<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
</configuration>"""
with open("/content/hadoop-3.5.0/etc/hadoop/core-site.xml", "w") as f:
    f.write(core_xml)

hdfs_xml = """<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.namenode.name.dir</name>
        <value>file:///content/hadoop_data/namenode</value>
    </property>
    <property>
        <name>dfs.datanode.data.dir</name>
        <value>file:///content/hadoop_data/datanode</value>
    </property>
</configuration>"""
with open("/content/hadoop-3.5.0/etc/hadoop/hdfs-site.xml", "w") as f:
    f.write(hdfs_xml)

mapred_xml = """<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>local</value>
    </property>
</configuration>"""
with open("/content/hadoop-3.5.0/etc/hadoop/mapred-site.xml", "w") as f:
    f.write(mapred_xml)


### 4. Format NameNode và khởi động NameNode, DataNode


In [ ]:
!hdfs namenode -format -force
!hdfs --daemon start namenode
!hdfs --daemon start datanode
!jps
!hdfs dfsadmin -report


### 5. Tạo thư mục HDFS và nạp dữ liệu đơn hàng (orders_2026_09.csv)


In [ ]:
!hdfs dfs -mkdir -p /user/student/ecommerce/input
!hdfs dfs -mkdir -p /user/student/ecommerce/output
!hdfs dfs -put -f orders_2026_09.csv /user/student/ecommerce/input/
!hdfs dfs -ls /user/student/ecommerce/input
!hdfs dfs -head /user/student/ecommerce/input/orders_2026_09.csv


### 6. Tạo mã nguồn Mapper (mapper_revenue.py) và Reducer (reducer_revenue.py)


In [ ]:
%%writefile mapper_revenue.py
import sys
import csv

reader = csv.DictReader(sys.stdin)
for row in reader:
    if row.get("status", "").strip() == "SUCCESS":
        category = row.get("category", "").strip()
        amount = row.get("amount", "").strip()
        if category and amount:
            print(f"{category}\t{amount}")


In [ ]:
%%writefile reducer_revenue.py
import sys

current_category = None
current_total = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split("\t")
    if len(parts) != 2:
        continue

    category, amount_str = parts
    try:
        amount = int(amount_str)
    except ValueError:
        continue

    if current_category == category:
        current_total += amount
    else:
        if current_category is not None:
            print(f"{current_category}\t{current_total}")
        current_category = category
        current_total = amount

if current_category is not None:
    print(f"{current_category}\t{current_total}")


### 7. Kiểm tra pipeline xử lý cục bộ


In [ ]:
!cat orders_2026_09.csv | python3 mapper_revenue.py | sort | python3 reducer_revenue.py


### 8. Thực thi Hadoop Streaming trên HDFS


In [ ]:
import glob
import subprocess

!hdfs dfs -rm -r -f /user/student/ecommerce/output/revenue_by_category

jar = glob.glob("/content/hadoop-3.5.0/share/hadoop/tools/lib/hadoop-streaming*.jar")[0]
cmd = [
    "hadoop", "jar", jar,
    "-files", "mapper_revenue.py,reducer_revenue.py",
    "-mapper", "python3 mapper_revenue.py",
    "-reducer", "python3 reducer_revenue.py",
    "-input", "/user/student/ecommerce/input/orders_2026_09.csv",
    "-output", "/user/student/ecommerce/output/revenue_by_category"
]
subprocess.run(cmd, check=True)


### 9. Đọc kết quả từ HDFS và trích xuất Top 3 nhóm sản phẩm có doanh thu cao nhất


In [ ]:
!hdfs dfs -ls /user/student/ecommerce/output/revenue_by_category
!hdfs dfs -cat /user/student/ecommerce/output/revenue_by_category/part-*
!hdfs dfs -get -f /user/student/ecommerce/output/revenue_by_category/part-* ./result_revenue_by_category.txt
!sort -k2 -nr result_revenue_by_category.txt | head -3


### 10. Mở rộng: Combiner tối ưu hóa truyền thông mạng


In [ ]:
%%writefile combiner_revenue.py
import sys

current_key = None
current_total = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split("\t")
    if len(parts) != 2:
        continue

    key, val_str = parts
    try:
        val = int(val_str)
    except ValueError:
        continue

    if current_key == key:
        current_total += val
    else:
        if current_key is not None:
            print(f"{current_key}\t{current_total}")
        current_key = key
        current_total = val

if current_key is not None:
    print(f"{current_key}\t{current_total}")


In [ ]:
!hdfs dfs -rm -r -f /user/student/ecommerce/output/revenue_with_combiner

cmd_combiner = [
    "hadoop", "jar", jar,
    "-files", "mapper_revenue.py,reducer_revenue.py,combiner_revenue.py",
    "-mapper", "python3 mapper_revenue.py",
    "-combiner", "python3 combiner_revenue.py",
    "-reducer", "python3 reducer_revenue.py",
    "-input", "/user/student/ecommerce/input/orders_2026_09.csv",
    "-output", "/user/student/ecommerce/output/revenue_with_combiner"
]
subprocess.run(cmd_combiner, check=True)
!hdfs dfs -cat /user/student/ecommerce/output/revenue_with_combiner/part-*


### 11. Mở rộng: Phân tích doanh thu theo Tỉnh/Thành phố


In [ ]:
%%writefile mapper_province.py
import sys
import csv

reader = csv.DictReader(sys.stdin)
for row in reader:
    if row.get("status", "").strip() == "SUCCESS":
        province = row.get("province", "").strip()
        amount = row.get("amount", "").strip()
        if province and amount:
            print(f"{province}\t{amount}")


In [ ]:
%%writefile reducer_province.py
import sys

current_province = None
current_total = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split("\t")
    if len(parts) != 2:
        continue

    province, amount_str = parts
    try:
        amount = int(amount_str)
    except ValueError:
        continue

    if current_province == province:
        current_total += amount
    else:
        if current_province is not None:
            print(f"{current_province}\t{current_total}")
        current_province = province
        current_total = amount

if current_province is not None:
    print(f"{current_province}\t{current_total}")


In [ ]:
!cat orders_2026_09.csv | python3 mapper_province.py | sort | python3 reducer_province.py | sort -k2 -nr


### 12. Mở rộng: Tính tỷ lệ đơn FAILED theo Kênh bán hàng (Channel)


In [ ]:
%%writefile mapper_channel_failed.py
import sys
import csv

reader = csv.DictReader(sys.stdin)
for row in reader:
    channel = row.get("channel", "").strip()
    status = row.get("status", "").strip()
    if channel and status:
        is_failed = 1 if status == "FAILED" else 0
        print(f"{channel}\t{is_failed}\t1")


In [ ]:
%%writefile reducer_channel_failed.py
import sys

current_channel = None
failed_count = 0
total_count = 0

for line in sys.stdin:
    line = line.strip()
    if not line:
        continue

    parts = line.split("\t")
    if len(parts) != 3:
        continue

    channel, failed_str, total_str = parts
    try:
        f = int(failed_str)
        t = int(total_str)
    except ValueError:
        continue

    if current_channel == channel:
        failed_count += f
        total_count += t
    else:
        if current_channel is not None:
            rate = (failed_count / total_count) * 100 if total_count > 0 else 0
            print(f"{current_channel}\t{failed_count}\t{total_count}\t{rate:.2f}%")
        current_channel = channel
        failed_count = f
        total_count = t

if current_channel is not None:
    rate = (failed_count / total_count) * 100 if total_count > 0 else 0
    print(f"{current_channel}\t{failed_count}\t{total_count}\t{rate:.2f}%")


In [ ]:
!cat orders_2026_09.csv | python3 mapper_channel_failed.py | sort | python3 reducer_channel_failed.py


### 13. Đối soát tính đúng đắn với Python cơ sở


In [ ]:
import csv
from collections import defaultdict

totals = defaultdict(int)
with open("orders_2026_09.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row.get("status", "").strip() == "SUCCESS":
            totals[row["category"].strip()] += int(row["amount"])

mapreduce_totals = {}
with open("result_revenue_by_category.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        cat, val = line.split("\t")
        mapreduce_totals[cat] = int(val)

print("Kiem tra khop 100% giua Python va MapReduce:", totals == mapreduce_totals)
